# Уровень 1. Ряды как модель накопления — с уклоном в ML для финтеха

## 1. Зачем вообще ряды в ML и финтехе

Ряд

$$
\sum_{n=1}^{\infty} u_n
$$

— это математическая модель накопления маленьких эффектов.

В чистой математике это сумма бесконечного числа слагаемых.

В ML и финтехе это почти всегда означает один из следующих смыслов:

- накопленная ошибка модели;
- накопленный риск;
- накопленный шум;
- накопленные обновления параметров;
- накопленное влияние старых наблюдений;
- накопленная прибыль/убыток;
- накопленный drift сигнала;
- накопленная величина штрафа или регуляризации.

То есть вопрос о сходимости ряда в прикладном смысле обычно звучит так:

- стабилизируется ли процесс;
- ограничен ли суммарный эффект;
- не взорвётся ли модель;
- не будет ли бесконечного накопления ошибки;
- не будет ли стратегия слишком чувствительной к шуму.

---

## 2. Главный перевод с языка матанализа на язык ML

Нужно научиться делать следующий перевод.

### Математика

Есть последовательность:

$$
u_1, u_2, u_3, \dots
$$

и есть ряд:

$$
u_1 + u_2 + u_3 + \dots
$$

### ML / финтех

- \(u_n\) — вклад шага \(n\);
- \(\sum_{k=1}^n u_k\) — накопленный эффект к моменту \(n\);
- сходимость ряда — суммарный эффект остаётся конечным;
- расходимость — суммарный эффект продолжает расти без ограничения.

---

## 3. Что означает сходимость и расходимость на интуитивном уровне

### Если ряд сходится

$$
\sum_{n=1}^{\infty} u_n < \infty
$$

это значит, что бесконечное число маленьких воздействий суммарно даёт конечный эффект.

Финтех-интерпретация:

- суммарная коррекция модели ограничена;
- накопленный риск под контролем;
- хвостовые вклады старых наблюдений затухают;
- алгоритм может стабилизироваться;
- шум со временем не разрушает систему.

### Если ряд расходится

$$
\sum_{n=1}^{\infty} u_n = \infty
$$

это значит, что даже маленькие вклады продолжают накапливаться так, что общая сумма не ограничена.

Финтех-интерпретация:

- сигнал или ошибка не затухают достаточно быстро;
- модель может продолжать дрейфовать;
- шум не усредняется полностью;
- стратегия или оценка может быть нестабильной;
- long-run effect бесконечен.

---

## 4. Почему это особенно важно в финтехе

В финтехе многие задачи имеют временную структуру:

- поток транзакций;
- поток котировок;
- поток событий клиента;
- обновление скоринга;
- онлайн-детекция фрода;
- кредитный риск по времени;
- drift признаков;
- decayed aggregation по истории.

Почти везде мы суммируем эффекты по времени:

$$
\text{aggregate} = \sum_{t} \text{contribution}_t
$$

Поэтому главный вопрос:

> как быстро должен убывать вклад прошлого, чтобы агрегат был устойчив?

Это и есть вопрос о сходимости ряда.

---

## 5. Базовые эталонные ряды, которые нужно знать наизусть

Это ядро. Их надо не просто помнить, а "чувствовать".

### 5.1 Геометрический ряд

$$
\sum_{n=0}^{\infty} q^n
$$

- сходится при \( |q| < 1 \);
- расходится при \( |q| \ge 1 \).

Интуиция:
каждый следующий вклад уменьшается в фиксированное число раз.

Финтех-смысл:
это естественная модель экспоненциального забывания прошлого.

Примеры:
- EMA;
- EWMA volatility;
- экспоненциальное сглаживание;
- discounting старых событий;
- weighting недавних транзакций сильнее старых.

Если веса равны

$$
1, q, q^2, q^3, \dots
$$

при \(0<q<1\), то суммарный вес истории конечен.

Это означает, что далёкое прошлое не может бесконечно доминировать.

---

### 5.2 p-ряд

$$
\sum_{n=1}^{\infty} \frac{1}{n^p}
$$

- сходится при \(p > 1\);
- расходится при \(p \le 1\).

Это один из самых важных фактов всей темы.

Интуиция:
вопрос не просто в том, что \(1/n^p \to 0\), а в том, **насколько быстро** это происходит.

Финтех-смысл:
если вклад старых наблюдений убывает как степень времени, то критический порог — это \(p=1\).

---

## 6. Три ключевых режима убывания и их прикладной смысл

### 6.1 Медленное убывание: \(1/n\)

$$
\sum_{n=1}^{\infty} \frac{1}{n}
$$

расходится.

Это гармонический ряд.

Интуиция:
члены убывают к нулю, но слишком медленно.

Почему это важно:
факт \(u_n \to 0\) сам по себе вообще не гарантирует сходимость.

Финтех-смысл:
если влияние прошлого затухает как \(1/t\), то общий накопленный эффект бесконечен.

Прикладная интерпретация:
- очень длинная память;
- старые события затухают слишком медленно;
- historical tail остаётся значимым слишком долго;
- возможен накопленный drift.

---

### 6.2 Достаточно быстрое убывание: \(1/n^2\)

$$
\sum_{n=1}^{\infty} \frac{1}{n^2}
$$

сходится.

Интуиция:
члены тоже стремятся к нулю, но уже достаточно быстро, чтобы общий вклад остался конечным.

Финтех-смысл:
если вклад старых транзакций, ошибок или обновлений убывает примерно как \(1/t^2\), то long-run accumulation ограничен.

Это часто интерпретируется как:
- история забывается достаточно быстро;
- хвост под контролем;
- накопленный эффект стабилизируется.

---

### 6.3 Очень быстрое убывание: \(q^n\), \(0<q<1\)

$$
\sum_{n=0}^{\infty} q^n
$$

сходится очень быстро.

Интуиция:
каждый следующий вклад в разы меньше предыдущего.

Финтех-смысл:
это режим агрессивного forgetting.

Примеры:
- экспоненциальные веса;
- риск-модель, ориентированная на недавние наблюдения;
- online updating с сильным предпочтением последних точек.

---

## 7. Частичные суммы — главный объект, а не просто члены ряда

Очень важно понимать:
ряд изучают не напрямую, а через его частичные суммы.

Для ряда

$$
\sum_{n=1}^{\infty} u_n
$$

частичная сумма:

$$
s_n = \sum_{k=1}^{n} u_k
$$

Именно \(s_n\) показывает, сколько уже накопилось к моменту \(n\).

Финтех-перевод:

- \(u_n\) — вклад очередной транзакции, дня, итерации;
- \(s_n\) — cumulative exposure / cumulative adjustment / cumulative risk.

Если \(s_n\) стремится к пределу, значит система стабилизируется по накопленному эффекту.

Если \(s_n\) растёт без границ, значит влияние продолжает накапливаться.

---

## 8. Визуальная интуиция: прямоугольники и площадь

Каждый член ряда можно представить как высоту прямоугольника ширины 1.

Тогда:

$$
u_1 + u_2 + \dots + u_n
$$

— это площадь первых \(n\) прямоугольников.

Если столбики быстро убывают, площадь может остаться конечной.

Если столбики убывают слишком медленно, площадь будет накапливаться бесконечно.

Финтех-перевод:

- каждый столбик — вклад очередного периода;
- общая площадь — total accumulated effect;
- вопрос сходимости — останется ли общая площадь ограниченной.

Это полезно для понимания:
- decayed feature aggregation;
- cumulative penalties;
- накопления хвостового риска;
- суммы discounted cash-flow weights.

---

## 9. Первый базовый финтех-кейс: decayed transaction signal

Представим, что мы строим признак для fraud detection:

$$
X_t = \sum_{k=1}^{t} w_k \cdot a_{t-k}
$$

где:
- \(a_{t-k}\) — событие или сумма транзакции из прошлого;
- \(w_k\) — вес события, произошедшего \(k\) шагов назад.

Тогда вопрос:
какие веса брать?

### Случай 1. \(w_k = 1/k\)

Тогда суммарный вес истории:

$$
\sum_{k=1}^{\infty} \frac{1}{k}
$$

расходится.

Что это значит:
- хвост очень тяжёлый;
- очень старые события всё ещё суммарно дают неограниченный вклад;
- feature может быть слишком «длиннопамятным».

### Случай 2. \(w_k = 1/k^2\)

Тогда:

$$
\sum_{k=1}^{\infty} \frac{1}{k^2}
$$

сходится.

Что это значит:
- история учитывается, но под контролем;
- очень старые транзакции почти не влияют;
- feature устойчивее.

### Случай 3. \(w_k = \lambda^k,\; 0<\lambda<1\)

Тогда веса затухают экспоненциально.

Что это значит:
- модель очень сильно ориентирована на недавние события;
- удобно для online fraud/risk;
- history window effectively finite.

---

## 10. Второй финтех-кейс: online model update

Пусть модель обновляется по одному наблюдению:

$$
\theta_{t+1} = \theta_t + \Delta_t
$$

Где \(\Delta_t\) — маленькое обновление.

Тогда после \(n\) шагов:

$$
\theta_n = \theta_0 + \sum_{t=1}^{n-1} \Delta_t
$$

То есть параметры модели — это начальное значение плюс ряд обновлений.

Если обновления слишком большие или убывают слишком медленно, параметры могут долго дрейфовать.

Если обновления контролируемы и суммарно конечны, модель может стабилизироваться.

Это прямой мост к SGD, но на Уровне 1 нам пока важно только увидеть:

> параметр модели — это накопленная сумма шагов.

---

## 11. Третий финтех-кейс: накопление ошибки прогноза

Пусть модель предсказывает вероятность дефолта, риска фрода или ожидаемую доходность, и на каждом шаге есть ошибка \(e_t\).

Тогда cumulative error:

$$
E_n = \sum_{t=1}^{n} e_t
$$

Если ошибки не затухают или их структура плохая, cumulative effect может расти без контроля.

В прикладном смысле это означает:
- model drift;
- unstable residual structure;
- long-horizon degradation.

С точки зрения рядов:
важно, насколько быстро убывают вклады ошибок во времени, если мы их reweight или accumulate.

---

## 12. Четвёртый финтех-кейс: discounting и present value

В финансах очень часто суммируют будущие cash flows с discounting:

$$
PV = \sum_{t=1}^{\infty} \frac{CF_t}{(1+r)^t}
$$

Если \(CF_t\) ограничены, а discount factor экспоненциальный, то это по сути геометрический тип затухания.

Почему это важно:
экспоненциальное затухание делает суммарную present value конечной.

Это один из самых естественных прикладных примеров сходящегося ряда.

---

## 13. Пять центральных мыслей, которые надо вынести

### Мысль 1
Не всякая последовательность, стремящаяся к нулю, даёт сходящийся ряд.

Пример:

$$
\frac{1}{n} \to 0,
$$

но

$$
\sum \frac{1}{n}
$$

расходится.

Для ML это означает:
маленькие по отдельности обновления всё ещё могут давать бесконечный cumulative effect.

---

### Мысль 2
Ключевой вопрос — скорость убывания.

Не важно только то, что вклад уменьшается.
Важно, насколько быстро он уменьшается.

---

### Мысль 3
Часто достаточно сравнить с эталоном.

Если твой процесс ведёт себя как:
- \(1/n\) — длинный хвост, расходимость;
- \(1/n^p,\; p>1\) — контролируемый хвост;
- \(q^n\) — агрессивное затухание.

---

### Мысль 4
Сходимость = bounded long-run accumulation.

Это один из самых прикладных смыслов всей темы.

---

### Мысль 5
Ряды — это язык анализа онлайн-процессов.

Любой streaming / iterative / temporal ML-процесс почти автоматически можно интерпретировать через накопление.

---

## 14. Что надо уметь после Уровня 1

После этого уровня ты должен без колебаний уметь:

1. Переводить ряд в прикладной смысл.
2. Отличать:
   - быстрое затухание,
   - медленное затухание,
   - критический режим.
3. Знать наизусть:
   - геометрический ряд;
   - p-ряды;
   - гармонический ряд как базовый контрпример.
4. Видеть в формулах типа
   $$
   \sum_t w_t x_t
   $$
   идею накопления с весами.
5. Объяснить, что означает сходимость в терминах:
   - памяти модели,
   - устойчивости,
   - long-run effect,
   - tail influence.

---

## 15. Мини-таблица для памяти

| Ряд | Поведение | Прикладной смысл |
|---|---|---|
| $$\sum 1/n$$ | расходится | хвост слишком тяжёлый, очень длинная память |
| $$\sum 1/n^2$$ | сходится | вклад прошлого достаточно быстро затухает |
| $$\sum q^n,\; 0<q<1$$ | сходится | экспоненциальное забывание, сильный акцент на недавних данных |

---

## 16. Практика именно под финтех-ML

### Блок A. Интерпретация
Для каждого ряда объяснить прикладной смысл:

1.
$$
\sum_{n=1}^{\infty} \frac{1}{n}
$$

2.
$$
\sum_{n=1}^{\infty} \frac{1}{n^2}
$$

3.
$$
\sum_{n=1}^{\infty} 0.95^n
$$

Вопросы:
- это длинная или короткая память?
- общий вес истории конечен или нет?
- как это повлияет на online signal?

---

### Блок B. Признаки как feature engineering
Рассмотреть признак:

$$
F_t = \sum_{k=1}^{t} w_k x_{t-k}
$$

и сравнить три варианта весов:
- \(w_k = 1/k\)
- \(w_k = 1/k^2\)
- \(w_k = 0.9^k\)

Для каждого объяснить:
- устойчивость;
- чувствительность к старым событиям;
- пригодность для fraud / risk / behavioral scoring.

---

### Блок C. Кумулятивные обновления
Пусть модель обновляет параметр на величину:
- \(1/t\)
- \(1/t^2\)
- \(0.95^t\)

Нужно ответить:
- суммарное движение параметра ограничено или нет?
- будет ли параметр потенциально дрейфовать бесконечно?
- какой режим лучше для стабильной онлайн-системы?

---

## 17. Кодовая практика, которую стоит сделать на Уровне 1

Нужно построить два типа графиков.

### 1. Графики самих членов
Для:
- \(1/n\)
- \(1/n^2\)
- \(0.9^n\)

чтобы увидеть скорость затухания.

### 2. Графики частичных сумм
Для тех же рядов:
- увидеть, кто стабилизируется;
- увидеть, кто продолжает расти.

Смысл:
именно график частичных сумм даёт интуицию накопления.

---

## 18. Финтех-мост к следующему уровню

Уровень 1 должен привести к следующему осознанию:

> learning rate, decayed weights, online updates, smoothing coefficients, discounted risk contributions —
> всё это по сути вопрос о том, как ведут себя соответствующие ряды.

То есть следующий шаг уже естественный:

- если веса = шаги обучения, то какие из них хороши;
- если есть шум, то как он накапливается;
- если есть онлайн-обновление, то когда оно стабильно.

Это и приведёт нас к Уровню 2: learning rate theory и к SGD.

---

## 19. Сверхкраткий итог уровня

Уровень 1 — это про одно:

> научиться видеть в любой временной или итеративной ML/финтех формуле накопление и сразу задавать вопрос:
> "этот хвост конечен или нет?"

Если конечен:
- влияние прошлого контролируемо,
- система имеет шанс быть устойчивой.

Если не конечен:
- у процесса длинная память,
- накопление может быть проблемным,
- шум, ошибка или drift могут жить слишком долго.